<a href="https://colab.research.google.com/github/zeynepaydin34/MicrochipStockAnalysis/blob/mainhttps%2Fdocs.github.com%2Frepositories%2Fconfiguring-branches-and-merges-in-your-repository%2Fmanaging-branches-in-your-repository%2Fchanging-the-default-branch/Technical_Analysis_Normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from google.colab import drive
import os
import joblib

drive.mount('/content/drive', force_remount=True)

hisseler = ['AMD', 'NVDA', 'ARM', 'INTC', 'MU']
ANA_KLASOR = '/content/drive/MyDrive/ChipStockAnalysis/TEKNİK ANALİZ VERİLERİ/'
MODEL_CIKTI_KLASORU = '/content/drive/MyDrive/ChipStockAnalysis/MODEL_VERİLERİ/'

sequence_length = 20     # Modelin geriye bakacağı gün sayısı
split_ratios = (0.70, 0.15, 0.15) # TRAIN / VALIDATION / TEST oranları
sigma_threshold = 3      # Aykırı değerleri kırpma eşiği (3-sigma)

if not os.path.exists(MODEL_CIKTI_KLASORU):
    os.makedirs(MODEL_CIKTI_KLASORU)
    print(f"✅ Yeni çıktı klasörü oluşturuldu: {MODEL_CIKTI_KLASORU}")

# Tüm hazır verilerin saklanacağı sözlük
all_data = {}

# -------------------------------
# 1) FONKSİYON: DİZİLEME (Sequencing)
# -------------------------------
def create_sequences(X, y, seq_length):
    X_seq, y_seq = [], []
    for i in range(seq_length, len(X)):
        X_seq.append(X[i-seq_length:i])
        y_seq.append(y[i])

    # y_seq'i (örnek, 1) şekline getiriyoruz
    return np.array(X_seq), np.array(y_seq).reshape(-1, 1)

# -------------------------------
# 2) TÜM HİSSELER İÇİN ANA DÖNGÜ
# -------------------------------
for hisse in hisseler:
    print(f"\n--- {hisse} için VERİ HAZIRLIĞI BAŞLADI ---")

    # --- Veri Yükleme
    INPUT_PATH = os.path.join(ANA_KLASOR, f'{hisse}_features_engineered.csv')
    df = pd.read_csv(INPUT_PATH, index_col=0, parse_dates=True)
    df.sort_index(inplace=True)

    feature_cols = [col for col in df.columns if col not in ['Symbol', 'Adj Close']]
    target_col = 'Adj Close'

    # --- Train/Validation/Test Split
    n = len(df)
    train_end = int(n * split_ratios[0]) # 70%
    val_end = int(n * (split_ratios[0] + split_ratios[1])) # 70% + 15% = 85%

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    # -------------------
    # 3) CLIPPING (Aykırı Değer Kırpma - Sadece Train setine)
    # -------------------
    print("   -> Aykırı değerler (3-sigma) kırpılıyor...")
    for col in feature_cols + [target_col]:
        mean = train_df[col].mean()
        std = train_df[col].std()
        lower = mean - sigma_threshold * std
        upper = mean + sigma_threshold * std
        train_df[col] = train_df[col].clip(lower, upper)

    # --- NumPy dizilerine çevirme
    X_train_data = train_df[feature_cols].values
    X_val_data = val_df[feature_cols].values
    X_test_data = test_df[feature_cols].values

    y_train_data = train_df[target_col].values.reshape(-1, 1)
    y_val_data = val_df[target_col].values.reshape(-1, 1)
    y_test_data = test_df[target_col].values.reshape(-1, 1)

    # -------------------
    # 4) SCALING (Ölçekleme - Fit sadece Train'e)
    # -------------------
    print("   -> Veriler ölçekleniyor (Standard/MinMax)...")

    # X Scaling (StandardScaler)
    X_scaler = StandardScaler()
    X_train_scaled = X_scaler.fit_transform(X_train_data)
    X_val_scaled = X_scaler.transform(X_val_data)
    X_test_scaled = X_scaler.transform(X_test_data)

    # Y Scaling (MinMaxScaler - 0-1 arası)
    y_scaler = MinMaxScaler(feature_range=(0, 1))
    y_train_scaled = y_scaler.fit_transform(y_train_data)
    y_val_scaled = y_scaler.transform(y_val_data) # Val setine transform
    y_test_scaled = y_scaler.transform(y_test_data) # Test setine transform

    # -------------------
    # 5) SEQUENCING (Dizileme)
    # -------------------
    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, sequence_length)
    X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled, sequence_length)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, sequence_length)

    print(f"✅ {hisse} Dizileme tamamlandı. Train: {X_train_seq.shape}, Val: {X_val_seq.shape}, Test: {X_test_seq.shape}")

    # -------------------
    # 6) KALICI KAYIT (Drive'a)
    # -------------------

    # NumPy Dizilerini Kaydetme (.npy)
    np.save(os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_X_train_seq.npy'), X_train_seq)
    np.save(os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_y_train_seq.npy'), y_train_seq)
    np.save(os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_X_val_seq.npy'), X_val_seq) # VAL KAYIT
    np.save(os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_y_val_seq.npy'), y_val_seq) # VAL KAYIT
    np.save(os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_X_test_seq.npy'), X_test_seq)
    np.save(os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_y_test_seq.npy'), y_test_seq)

    # Scaler Nesnelerini Kaydetme (.pkl)
    joblib.dump(X_scaler, os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_X_scaler.pkl'))
    joblib.dump(y_scaler, os.path.join(MODEL_CIKTI_KLASORU, f'{hisse}_y_scaler.pkl'))

    print(f"   -> Veriler ve Scaler nesneleri Drive'a kaydedildi.")

print("\n🎉 Tüm hisseler için Teknik Veri Ön İşleme (Train/Val/Test) başarıyla tamamlandı.")

Mounted at /content/drive
✅ Yeni çıktı klasörü oluşturuldu: /content/drive/MyDrive/ChipStockAnalysis/MODEL_VERİLERİ/

--- AMD için VERİ HAZIRLIĞI BAŞLADI ---
   -> Aykırı değerler (3-sigma) kırpılıyor...
   -> Veriler ölçekleniyor (Standard/MinMax)...
✅ AMD Dizileme tamamlandı. Train: (4659, 20, 24), Val: (983, 20, 24), Test: (983, 20, 24)
   -> Veriler ve Scaler nesneleri Drive'a kaydedildi.

--- NVDA için VERİ HAZIRLIĞI BAŞLADI ---
   -> Aykırı değerler (3-sigma) kırpılıyor...
   -> Veriler ölçekleniyor (Standard/MinMax)...
✅ NVDA Dizileme tamamlandı. Train: (4659, 20, 24), Val: (983, 20, 24), Test: (983, 20, 24)
   -> Veriler ve Scaler nesneleri Drive'a kaydedildi.

--- ARM için VERİ HAZIRLIĞI BAŞLADI ---
   -> Aykırı değerler (3-sigma) kırpılıyor...
   -> Veriler ölçekleniyor (Standard/MinMax)...
✅ ARM Dizileme tamamlandı. Train: (301, 20, 24), Val: (49, 20, 24), Test: (49, 20, 24)
   -> Veriler ve Scaler nesneleri Drive'a kaydedildi.

--- INTC için VERİ HAZIRLIĞI BAŞLADI ---
   ->